<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 220px; height: 150px; vertical-align: middle;">
            <img src="../assets/aaa.png" width="220" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">自主交易员</h2>
            <span style="color:#ff7800;">一个股票交易模拟，展示由 MCP Server 的工具和资源驱动的自主 Agent。
            </span>
        </td>
    </tr>
</table>

### 第六周第四天

现在 — 介绍毕业项目：


# 自主交易员

一个股票交易模拟，包含 4 个交易员和 1 个研究员，由一系列 MCP Server 的工具和资源驱动：

1. 我们自制的 Accounts MCP Server（由我们的工程团队编写！）
2. Fetch（通过本地无头浏览器获取网页）
3. Memory（记忆）
4. Brave Search（搜索）
5. Financial data（金融数据）

以及用于读取交易员账户信息和投资策略的资源。

今天实验的目标是创建一个新的 Python 模块 `traders.py`，用来管理交易大厅中的单个交易员。

我们将在实验中进行探索和试验，然后在准备好后迁移到 Python 模块中。


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">再次提醒 --</h2>
            <span style="color:#ff7800;">请不要将此用于实际的交易决策！！
            </span>
        </td>
    </tr>
</table>

In [ ]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace, Tool
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display
from datetime import datetime
from accounts_client import read_accounts_resource, read_strategy_resource
from accounts import Account

load_dotenv(override=True)

### 让我们先收集交易员的 MCP 参数

In [ ]:
polygon_api_key = os.getenv("POLYGON_API_KEY")
polygon_plan = os.getenv("POLYGON_PLAN")

is_paid_polygon = polygon_plan == "paid"
is_realtime_polygon = polygon_plan == "realtime"

print(is_paid_polygon)
print(is_realtime_polygon)

In [3]:
if is_paid_polygon or is_realtime_polygon:
    market_mcp = {"command": "uvx","args": ["--from", "git+https://github.com/polygon-io/mcp_polygon@master", "mcp_polygon"], "env": {"POLYGON_API_KEY": polygon_api_key}}
else:
    market_mcp = ({"command": "uv", "args": ["run", "market_server.py"]})

trader_mcp_server_params = [
    {"command": "uv", "args": ["run", "accounts_server.py"]},
    {"command": "uv", "args": ["run", "push_server.py"]},
    market_mcp
]

### 现在是我们的研究员

In [4]:
brave_env = {"BRAVE_API_KEY": os.getenv("BRAVE_API_KEY")}

researcher_mcp_server_params = [
    {"command": "uvx", "args": ["mcp-server-fetch"]},
    {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": brave_env}
]

### 现在为每个创建 MCPServerStdio

In [13]:
researcher_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in researcher_mcp_server_params]
trader_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in trader_mcp_server_params]
mcp_servers = trader_mcp_servers + researcher_mcp_servers

### 现在让我们创建一个研究员 Agent 来进行市场调研

并将其转换为工具 — 还记得这在 OpenAI Agents SDK 中是如何工作的吗？以及与 handoff 的区别？

In [ ]:
async def get_researcher(mcp_servers) -> Agent:
    instructions = f"""你是一名金融研究员。你能够在网上搜索有趣的金融新闻，
寻找可能的交易机会，并协助研究。
根据请求，你进行必要的研究并回复你的发现。
花时间进行多次搜索以获得全面的概述，然后总结你的发现。
如果没有特定的请求，就根据搜索最新新闻回复投资机会。
当前日期时间是 {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""
    researcher = Agent(
        name="Researcher",
        instructions=instructions,
        model="gpt-4.1-mini",
        mcp_servers=mcp_servers,
    )
    return researcher

In [ ]:
async def get_researcher_tool(mcp_servers) -> Tool:
    researcher = await get_researcher(mcp_servers)
    return researcher.as_tool(
            tool_name="Researcher",
            tool_description="此工具在线研究新闻和机会，\
                可以根据你对特定股票的研究请求，\
                或者搜索值得关注的金融新闻和机会。\
                请描述你在寻找什么样的研究。"
        )

In [ ]:
research_question = "亚马逊的最新消息是什么？"

for server in researcher_mcp_servers:
    await server.connect()
researcher = await get_researcher(researcher_mcp_servers)
with trace("Researcher"):
    result = await Runner.run(researcher, research_question, max_turns=30)
display(Markdown(result.final_output))


### 查看 Trace 追踪

https://platform.openai.com/traces

In [ ]:
ed_initial_strategy = "你是一名日内交易员，根据新闻和市场情况积极买卖股票。"
Account.get("Ed").reset(ed_initial_strategy)

display(Markdown(await read_accounts_resource("Ed")))
display(Markdown(await read_strategy_resource("Ed")))

### 现在 — 创建我们的交易员 Agent

In [ ]:
agent_name = "Ed"

# 使用 MCP Server 读取资源
account_details = await read_accounts_resource(agent_name)
strategy = await read_strategy_resource(agent_name)

instructions = f"""
你是一名管理股票投资组合的交易员。你的名字是 {agent_name}，你的账户名也是 {agent_name}。
你可以使用工具在网上搜索公司新闻、查看股价以及买卖股票。
你的投资组合投资策略是：
{strategy}
你当前的持仓和余额是：
{account_details}
你有执行网络搜索相关新闻和信息的工具。
你有查看股价的工具。
你有买卖股票的工具。
你有保存公司信息、研究和思考记忆的工具。
请使用这些工具来管理你的投资组合。在你认为合适时进行交易；不要等待指令或请求确认。
"""

prompt = """
使用你的工具来对你的投资组合做出决策。
调查新闻和市场，做出决定，执行交易，然后回复你的操作摘要。
"""

In [ ]:
print(instructions)

### 运行我们的交易员

In [ ]:
for server in mcp_servers:
    await server.connect()

researcher_tool = await get_researcher_tool(researcher_mcp_servers)
trader = Agent(
    name=agent_name,
    instructions=instructions,
    tools=[researcher_tool],
    mcp_servers=trader_mcp_servers,
    model="gpt-4o-mini",
)
with trace(agent_name):
    result = await Runner.run(trader, prompt, max_turns=30)
display(Markdown(result.final_output))

### 然后去查看 Trace 追踪

http://platform.openai.com/traces


In [ ]:
# 让我们看看交易的结果

await read_accounts_resource(agent_name)

### 现在是时候回顾由此制作的 Python 模块了：

`mcp_params.py` 是指定 MCP Server 的地方。你会注意到我引入了一些熟悉的朋友：memory 和 push notifications！

`templates.py` 是设置指令和消息的地方（即系统提示词和用户提示词）

`traders.py` 将所有内容整合在一起。

你会注意到我用了一些比较巧妙的代码：

```
async with AsyncExitStack() as stack:
    mcp_servers = [await stack.enter_async_context(MCPServerStdio(params)) for params in mcp_server_params]
```

这只是一种整洁的方式来组合我们的 "with" 语句（称为上下文管理器），这样我们就不需要写这样丑陋的代码：

```
async with MCPServerStdio(params=params1) as mcp_server1:
    async with MCPServerStdio(params=params2) as mcp_server2:
        async with MCPServerStdio(params=params3) as mcp_server3:
            mcp_servers = [mcp_server1, mcp_server2, mcp_server3]
```

但它们是等价的。


In [2]:
from traders import Trader


In [3]:
trader = Trader("Ed")

In [ ]:
await trader.run()

In [ ]:
await read_accounts_resource("Ed")

### 现在查看 Trace 追踪

https://platform.openai.com/traces

### 我们总共使用了多少个工具？

In [ ]:
from mcp_params import trader_mcp_server_params, researcher_mcp_server_params

all_params = trader_mcp_server_params + researcher_mcp_server_params("ed")

count = 0
for each_params in all_params:
    async with MCPServerStdio(params=each_params, client_session_timeout_seconds=60) as server:
        mcp_tools = await server.list_tools()
        count += len(mcp_tools)
print(f"我们有 {len(all_params)} 个 MCP Server，和 {count} 个工具")